In [0]:
%sql
-- Tabla Gold 1: KPIs operativos y financieros por vehículo/activo
CREATE OR REPLACE TABLE logistics_prod.gold.asset_performance_summary
CLUSTER BY (asset_id)
AS
SELECT
    asset_id,
    COUNT(*) AS total_readings,
    ROUND(AVG(temperature_c), 2) AS avg_temp_c,
    ROUND(MAX(temperature_c), 2) AS max_temp_c,
    ROUND(AVG(humidity_pct), 2) AS avg_humidity_pct,
    ROUND(AVG(waiting_time_mins), 1) AS avg_waiting_mins,
    SUM(is_delayed) AS total_delayed_events,
    ROUND(SUM(is_delayed) * 100.0 / COUNT(*), 2) AS delay_rate_pct,
    ROUND(AVG(asset_utilization_pct), 2) AS avg_utilization_pct,
    ROUND(SUM(transaction_amount), 2) AS total_cargo_value
FROM logistics_prod.silver.logistics_clean
GROUP BY asset_id;

-- Tabla Gold 2: Alertas operativas críticas (Excursiones térmicas y retrasos por tráfico)
CREATE OR REPLACE TABLE logistics_prod.gold.operational_risk_alerts
CLUSTER BY (risk_level, asset_id)
AS
SELECT
    event_timestamp,
    asset_id,
    shipment_status,
    temperature_c,
    humidity_pct,
    traffic_status,
    waiting_time_mins,
    delay_reason,
    transaction_amount,
    CASE
        WHEN temperature_c > 25.0 AND is_delayed = 1 THEN 'CRITICAL_TEMP_AND_DELAY'
        WHEN temperature_c > 25.0 THEN 'HIGH_TEMPERATURE_EXCURSION'
        WHEN is_delayed = 1 AND traffic_status = 'Heavy' THEN 'MEDIUM_TRAFFIC_DELAY'
        ELSE 'LOW_RISK'
    END AS risk_level
FROM logistics_prod.silver.logistics_clean
WHERE temperature_c > 25.0 OR is_delayed = 1;
